# Kalman Filter — Market Data Analysis

Having validated the filter on synthetic data we now apply it to real market prices.
Key questions:

- Does the smoothed estimate track price without obvious lag on trend changes?
- How sensitive is the output to Q/R tuning?
- What does the 2-D filter's extracted *trend* component look like?


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from kalman_filter import KalmanFilter1D, KalmanFilter2D
from filters import load_asset, DATA_DIR, ASSETS, COLORS

plt.rcParams['figure.dpi'] = 110

TICKERS = ['SPY', 'GLD', 'USO', 'CPER']
data = {}
for ticker in TICKERS:
    df = load_asset(ticker, DATA_DIR)
    if df is not None:
        data[ticker] = df['Close']
        print(f'{ticker}: {len(df)} days  '
              f'({df.index[0].date()} → {df.index[-1].date()})')


In [ ]:
# ── Apply KalmanFilter1D to each asset ─────────────────────────────────────
# R is scaled to the observed daily variance so the filter is not blind to
# each asset's actual noise level.
Q_RATIO = 0.01   # Q = Q_RATIO * daily variance
R_RATIO = 2.0    # R = R_RATIO * daily variance

all_tickers = list(ASSETS.keys())
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    if ticker not in data:
        continue
    ax   = axes[i]
    meas = data[ticker].values
    idx  = data[ticker].index
    dvar = float(np.var(np.diff(meas)))

    kf  = KalmanFilter1D(Q=dvar*Q_RATIO, R=dvar*R_RATIO, x0=meas[0])
    est, _, _ = kf.filter(meas)

    color = COLORS[all_tickers.index(ticker)] if ticker in all_tickers else '#2196F3'
    ax.plot(idx, meas, color='#9E9E9E', lw=0.7, alpha=0.55, label='Raw price')
    ax.plot(idx, est,  color=color,    lw=1.5,             label='Kalman estimate')
    ax.set_title(f'{ticker}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Price (USD)')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('KalmanFilter1D Applied to Market Prices', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()


## Q/R Tuning — Smoothness vs Responsiveness

The Q/R ratio is the single most important parameter.  Below we fix R and vary Q
on SPY to show the range from over-smoothed to under-smoothed.


In [ ]:
spy_meas = data['SPY'].values
spy_idx  = data['SPY'].index
dvar_spy = float(np.var(np.diff(spy_meas)))
R_SPY    = dvar_spy * 2.0

tune_configs = [
    {'Q': dvar_spy * 0.001, 'color': '#4CAF50', 'label': 'Q×0.001  (smooth / lagging)'},
    {'Q': dvar_spy * 0.01,  'color': '#FF5722', 'label': 'Q×0.01   (balanced)'},
    {'Q': dvar_spy * 0.5,   'color': '#9C27B0', 'label': 'Q×0.5    (responsive / noisy)'},
]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(spy_idx, spy_meas, color='#9E9E9E', lw=0.7, alpha=0.45, label='SPY raw')
for cfg in tune_configs:
    kf  = KalmanFilter1D(Q=cfg['Q'], R=R_SPY, x0=spy_meas[0])
    est, _, _ = kf.filter(spy_meas)
    ax.plot(spy_idx, est, color=cfg['color'], lw=1.4, label=cfg['label'])

ax.set_title('SPY — Kalman Filter Q/R Tuning (R fixed)', fontsize=12)
ax.set_ylabel('Price (USD)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2-D Kalman Filter — Extracting the Trend Component

The 2-D filter estimates both price and its local trend simultaneously.  The trend
state can be interpreted as a momentum signal: positive → price accelerating up,
negative → accelerating down.  This is something a 1-D filter cannot produce.


In [ ]:
Q2_spy = np.diag([dvar_spy * 0.5, dvar_spy * 1e-4])
R2_spy = dvar_spy * 100.0

kf2 = KalmanFilter2D(Q=Q2_spy, R=R2_spy, x0=np.array([spy_meas[0], 0.0]))
prices2d, trends2d, _ = kf2.filter(spy_meas)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax1.plot(spy_idx, spy_meas,  color='#9E9E9E', lw=0.8, alpha=0.5, label='Raw price')
ax1.plot(spy_idx, prices2d,  color='#2196F3', lw=1.6,            label='KF2D price estimate')
ax1.set_ylabel('Price (USD)')
ax1.set_title('SPY — KalmanFilter2D: Price Estimate', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Express trend as percent-per-day for interpretability
daily_pct = np.where(prices2d > 0, trends2d / prices2d * 100, 0.0)
ax2.plot(spy_idx, daily_pct, color='#FF9800', lw=1.0)
ax2.fill_between(spy_idx, daily_pct, alpha=0.2, color='#FF9800')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_ylabel('Trend (% / day)')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
